# 02 — Fmr1 | Long-running behavioral alignment and electrophysiology

This is the **full Fmr1 version** of the WT pipeline, using the same event definitions, window exclusions, adaptive logISI threshold, burst detection and mouse-level aggregation. It reads original data through the IBL ONE API and saves outputs to `Behavior_aligned/04_results/Fmr1/`.

Run the cells **in order**: (1) environment and ONE; (2) derive behavior windows for all 54 sessions (resumable); (3) QC/filter windows; (4) compute spikes for Fmr1 neurons (expensive and resumable). A pilot of five Fmr1 sessions already confirmed trial and wheel-data access.

The output of stage 3 can be analyzed in `Fmr1/03_analysis_heatmaps.ipynb`. Do not delete the per-session cache when resuming. Processing may be long due to probe downloads and per-window measurements.


In [ ]:
# Runtime setup — run at the start of a fresh Colab session
%pip -q install one-api ibllib iblatlas pandas numpy scipy matplotlib statsmodels pyarrow
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from one.api import ONE
import pandas as pd
import numpy as np
ROOT = Path('/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned')
assert ROOT.exists(), f'Cannot find project root: {ROOT}'
one = ONE(base_url='https://openalyx.internationalbrainlab.org', username='intbrainlab', password='international', silent=True)
print('Project root:', ROOT)


## Stage 1 — Generate Fmr1 behavioral windows (resumable)


In [ ]:

# ============================================================
# BEHAVIOR-ALIGNED ELECTROPHYSIOLOGY
# STEP 2 — BUILD FIVE BEHAVIORAL WINDOW TYPES FOR ALL Fmr1
#
# Q = nominal pre-stimulus quiescence
# S = stimulus onset to first detected wheel movement
# M = actual wheel movement intervals before response
# F = feedback onset + 0.5 s (reward/error separated)
# I = post-feedback interval until next trial quiescence
#
# This cell creates behavioral timestamps only.
# No spikes or electrophysiological metrics are processed.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

META_DIR = ROOT / "02_session_metadata"
RESULTS_DIR = ROOT / "04_results" / "Fmr1"
FIGURES_DIR = ROOT / "05_figures" / "Fmr1" / "Pilot_timelines" / "PNG"

SESSION_DIR = RESULTS_DIR / "session_windows"

for folder in (RESULTS_DIR, FIGURES_DIR, SESSION_DIR):
    folder.mkdir(parents=True, exist_ok=True)

FEEDBACK_WINDOW_S = 0.5

PHASES = [
    "Q_QUIESCENCE",
    "S_STIM_PREMOVEMENT",
    "M_WHEEL_MOVEMENT",
    "F_FEEDBACK",
    "I_INTERTRIAL"
]

# ------------------------------------------------------------
# 1. LOAD EXISTING Fmr1 SESSION INVENTORY
# ------------------------------------------------------------

inventory_path = META_DIR / "mouse_probe_inventory_recovered.csv"

if not inventory_path.exists():
    raise FileNotFoundError(
        "Recovered inventory missing. "
        "Run the setup from the previous notebook section."
    )

inventory = pd.read_csv(inventory_path)

fmr_sessions = (
    inventory.loc[
        inventory["genotype"] == "Fmr1",
        ["eid", "subject"]
    ]
    .dropna()
    .drop_duplicates("eid")
    .reset_index(drop=True)
)

print("Fmr1 mice:", fmr_sessions["subject"].nunique())
print("Fmr1 sessions:", len(fmr_sessions))


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def get_values(obj, key, n, fill=np.nan):
    """Return a numeric trial-length array."""
    if key not in obj:
        return np.full(n, fill, dtype=float)

    x = np.asarray(obj[key])

    if x.ndim != 1 or len(x) != n:
        return np.full(n, fill, dtype=float)

    return pd.to_numeric(
        pd.Series(x),
        errors="coerce"
    ).to_numpy(dtype=float)


def safe_windows(obj, key):
    """Return sorted N x 2 movement intervals."""
    if key not in obj:
        return np.empty((0, 2), dtype=float)

    x = np.asarray(obj[key], dtype=float)

    if x.size == 0:
        return np.empty((0, 2), dtype=float)

    if x.ndim != 2 or x.shape[1] != 2:
        raise ValueError("Invalid wheel movement intervals.")

    x = x[
        np.isfinite(x).all(axis=1) &
        (x[:, 1] > x[:, 0])
    ]

    return x[np.argsort(x[:, 0])]


def overlap_duration(start, end, intervals):
    """Total movement overlap, with overlapping spans merged."""
    if end <= start or len(intervals) == 0:
        return 0.0

    clips = []
    for a, b in intervals:
        lo, hi = max(a, start), min(b, end)
        if hi > lo:
            clips.append((lo, hi))

    if not clips:
        return 0.0

    clips.sort()
    merged = [list(clips[0])]

    for a, b in clips[1:]:
        if a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])

    return sum(b-a for a, b in merged)


def add_window(
    output, eid, subject, trial_id, phase,
    start, end, feedback_type,
    movement_overlap_s=0.0,
    note=""
):
    """Append one valid phase interval."""
    if not (
        np.isfinite(start) and
        np.isfinite(end) and
        end > start
    ):
        return

    output.append({
        "eid": eid,
        "subject": subject,
        "genotype": "Fmr1",
        "trial_id": trial_id,
        "phase": phase,
        "start_s": float(start),
        "end_s": float(end),
        "duration_s": float(end-start),
        "feedback_type": feedback_type,
        "movement_overlap_s": movement_overlap_s,
        "note": note
    })


# ------------------------------------------------------------
# 3. PROCESS ONE SESSION
# ------------------------------------------------------------

def process_session(eid, subject):

    trials = one.load_object(
        str(eid),
        "trials",
        collection="alf"
    )

    moves = one.load_object(
        str(eid),
        "wheelMoves",
        collection="alf"
    )

    intervals = np.asarray(
        trials["intervals"],
        dtype=float
    )

    if intervals.ndim != 2 or intervals.shape[1] != 2:
        raise ValueError("Invalid trial intervals.")

    n = len(intervals)

    start = intervals[:, 0]
    end = intervals[:, 1]

    stim = get_values(trials, "stimOn_times", n)
    first = get_values(trials, "firstMovement_times", n)
    response = get_values(trials, "response_times", n)
    feedback = get_values(trials, "feedback_times", n)
    feedback_type = get_values(trials, "feedbackType", n)
    q_period = get_values(trials, "quiescencePeriod", n)

    wheel = safe_windows(moves, "intervals")

    output = []
    audit = []

    # Nominal quiescence starts for all trials.
    q_start = np.full(n, np.nan)

    for i in range(n):

        if (
            np.isfinite(stim[i]) and
            np.isfinite(q_period[i]) and
            q_period[i] > 0
        ):
            q_start[i] = max(
                start[i],
                stim[i] - q_period[i]
            )

    # Iterate through all experimental trials.
    for i in range(n):

        trial_start = start[i]
        trial_end = end[i]
        stim_on = stim[i]
        response_t = response[i]
        feedback_t = feedback[i]

        fb = feedback_type[i]

        if fb == 1:
            outcome = "correct_reward"
        elif fb == -1:
            outcome = "error_feedback"
        else:
            outcome = "unknown"

        trial_moves = wheel[
            (wheel[:, 0] < trial_end) &
            (wheel[:, 1] > trial_start)
        ]

        # --------------------------------------------
        # Q — Nominal quiescence before stimulus onset
        # --------------------------------------------

        if np.isfinite(q_start[i]):

            q0 = q_start[i]
            q1 = stim_on

            overlap = overlap_duration(
                q0, q1, trial_moves
            )

            add_window(
                output, eid, subject, i, PHASES[0],
                q0, q1, outcome,
                movement_overlap_s=overlap,
                note=(
                    "wheel_overlap_detected"
                    if overlap > 0
                    else "nominal_quiescence"
                )
            )

        # --------------------------------------------
        # S — Stimulus onset until first wheel movement
        # --------------------------------------------

        # A movement that is already active at stimulus onset
        # creates no movement-free premovement interval.
        if np.isfinite(stim_on):

            response_limit = (
                response_t
                if np.isfinite(response_t)
                and response_t > stim_on
                else (
                    feedback_t
                    if np.isfinite(feedback_t)
                    and feedback_t > stim_on
                    else trial_end
                )
            )

            eligible_moves = trial_moves[
                (trial_moves[:, 1] > stim_on) &
                (trial_moves[:, 0] < response_limit)
            ]

            if len(eligible_moves):
                movement_onset = max(
                    stim_on,
                    np.min(eligible_moves[:, 0])
                )
            else:
                movement_onset = response_limit

            s_end = min(
                movement_onset,
                response_limit,
                trial_end
            )

            add_window(
                output, eid, subject, i, PHASES[1],
                stim_on, s_end, outcome,
                note="stimulus_before_detected_movement"
            )

        # --------------------------------------------
        # M — Actual movement intervals until response
        # --------------------------------------------

        if (
            np.isfinite(stim_on) and
            np.isfinite(response_t) and
            response_t > stim_on
        ):

            motor_end = min(response_t, trial_end)

            # Merge overlapping wheel movement intervals
            # to prevent double counting movement time.
            clipped = []

            for a, b in trial_moves:

                lo = max(a, stim_on)
                hi = min(b, motor_end)

                if hi > lo:
                    clipped.append([lo, hi])

            clipped.sort()

            merged = []

            for a, b in clipped:
                if merged and a <= merged[-1][1]:
                    merged[-1][1] = max(
                        merged[-1][1], b
                    )
                else:
                    merged.append([a, b])

            for a, b in merged:

                add_window(
                    output, eid, subject, i, PHASES[2],
                    a, b, outcome,
                    note="actual_wheel_movement"
                )

        # --------------------------------------------
        # F — Feedback response (first 500 ms)
        # --------------------------------------------

        next_q_start = (
            q_start[i+1] if i+1 < n
            else np.nan
        )

        if np.isfinite(feedback_t):

            feedback_end = feedback_t + FEEDBACK_WINDOW_S

            # Prevent overlap with next trial quiescence.
            if np.isfinite(next_q_start):
                feedback_end = min(
                    feedback_end, next_q_start
                )

            add_window(
                output, eid, subject, i, PHASES[3],
                feedback_t, feedback_end, outcome,
                note="feedback_locked_500ms"
            )

        # --------------------------------------------
        # I — Post-feedback inter-trial waiting
        # --------------------------------------------

        if (
            i < n-1 and
            np.isfinite(feedback_t) and
            np.isfinite(next_q_start)
        ):

            iti_start = feedback_t + FEEDBACK_WINDOW_S

            add_window(
                output, eid, subject, i, PHASES[4],
                iti_start, next_q_start, outcome,
                note="post_feedback_to_next_quiescence"
            )

        audit.append({
            "eid": eid,
            "subject": subject,
            "trial_id": i,
            "trial_duration_s": trial_end-trial_start,
            "stim_present": bool(np.isfinite(stim_on)),
            "first_movement_present": bool(np.isfinite(first[i])),
            "premature_first_movement": bool(
                np.isfinite(first[i]) and
                np.isfinite(stim_on) and
                first[i] < stim_on
            ),
            "feedback_present": bool(np.isfinite(feedback_t)),
            "quiescence_present": bool(np.isfinite(q_start[i])),
            "outcome": outcome,
            "wheel_movement_count": len(trial_moves)
        })

    return pd.DataFrame(output), pd.DataFrame(audit)


# ------------------------------------------------------------
# 4. PROCESS ALL Fmr1 SESSIONS — RESUMABLE
# ------------------------------------------------------------

session_reports = []

for number, record in enumerate(
    fmr_sessions.itertuples(index=False), start=1
):

    eid = str(record.eid)
    subject = str(record.subject)

    windows_path = SESSION_DIR / f"{eid}_windows.csv"
    audit_path = SESSION_DIR / f"{eid}_audit.csv"

    try:

        if windows_path.exists() and audit_path.exists():

            windows_df = pd.read_csv(windows_path)
            audit_df = pd.read_csv(audit_path)
            status = "cached"

        else:

            windows_df, audit_df = process_session(
                eid, subject
            )

            if windows_df.empty:
                raise RuntimeError("No valid windows created.")

            windows_df.to_csv(
                windows_path, index=False
            )
            audit_df.to_csv(
                audit_path, index=False
            )

            status = "processed"

        session_reports.append({
            "eid": eid,
            "subject": subject,
            "status": status,
            "n_trials": len(audit_df),
            "n_windows": len(windows_df)
        })

        print(
            f"[{number}/{len(fmr_sessions)}] "
            f"{subject} | {status} | "
            f"{len(audit_df)} trials | "
            f"{len(windows_df)} windows"
        )

    except Exception as exc:

        session_reports.append({
            "eid": eid,
            "subject": subject,
            "status": "error",
            "error": str(exc)[:300]
        })

        print(
            f"[{number}/{len(fmr_sessions)}] "
            f"{subject} ERROR: {str(exc)[:150]}"
        )

report = pd.DataFrame(session_reports)

report.to_csv(
    RESULTS_DIR / "Fmr1_behavior_processing_report.csv",
    index=False
)

# ------------------------------------------------------------
# 5. COMBINE SUCCESSFULLY PROCESSED SESSIONS
# ------------------------------------------------------------

all_windows = []
all_audits = []

for r in session_reports:

    if r["status"] not in ("cached", "processed"):
        continue

    eid = r["eid"]

    all_windows.append(
        pd.read_csv(
            SESSION_DIR / f"{eid}_windows.csv"
        )
    )

    all_audits.append(
        pd.read_csv(
            SESSION_DIR / f"{eid}_audit.csv"
        )
    )

if not all_windows:
    raise RuntimeError("No Fmr1 sessions successfully processed.")

windows = pd.concat(
    all_windows, ignore_index=True
)

audits = pd.concat(
    all_audits, ignore_index=True
)

windows.to_csv(
    RESULTS_DIR / "Fmr1_all_behavior_windows.csv",
    index=False
)

audits.to_csv(
    RESULTS_DIR / "Fmr1_all_trial_audit.csv",
    index=False
)

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Fmr1 BEHAVIORAL WINDOW SUMMARY")
print("=" * 70)

print("Mice:", windows["subject"].nunique())
print("Sessions:", windows["eid"].nunique())
print("Trials:", len(audits))
print("Windows:", len(windows))

summary = (
    windows.groupby("phase")["duration_s"]
    .agg(["count", "sum", "mean", "median", "min", "max"])
    .reindex(PHASES)
    .round(4)
)

display(summary)

summary.to_csv(
    RESULTS_DIR / "Fmr1_phase_duration_summary.csv"
)

print("\nTRIAL AUDIT")
display(
    audits[[
        "stim_present",
        "first_movement_present",
        "premature_first_movement",
        "feedback_present",
        "quiescence_present"
    ]].sum().to_frame("count")
)

print("\nFEEDBACK TYPES")
display(audits["outcome"].value_counts())

print("\nQUIESCENCE MOVEMENT OVERLAP")
q = windows.loc[
    windows["phase"] == "Q_QUIESCENCE"
]

print(
    "Windows containing wheel movements:",
    int((q["movement_overlap_s"] > 0).sum()),
    "of", len(q)
)

print("\nSESSION PROCESSING REPORT")
display(report["status"].value_counts())


# ------------------------------------------------------------
# 7. PLOT PHASE DURATION DISTRIBUTIONS
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 5,
    figsize=(18, 4.5),
    layout="constrained"
)

colors = [
    "#8DA0CB",
    "#66C2A5",
    "#FC8D62",
    "#B48BC8",
    "#9AABB9"
]

for ax, phase, color in zip(axes, PHASES, colors):

    durations = windows.loc[
        windows["phase"] == phase,
        "duration_s"
    ]

    ax.hist(
        durations,
        bins=40,
        color=color,
        edgecolor="white"
    )

    ax.set_title(
        phase.split("_", 1)[1],
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel("Duration (s)")
    ax.set_ylabel("Windows")
    ax.grid(axis="y", alpha=0.15)

fig.suptitle(
    "Fmr1 mice — Behavioral window duration distributions",
    fontsize=15,
    fontweight="bold"
)

fig.savefig(
    FIGURES_DIR / "Fmr1_behavior_window_durations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nSaved results to:", RESULTS_DIR)
print("Saved figures to:", FIGURES_DIR)


## Stage 2 — Flag anomalous behavioral windows (same WT thresholds)


In [ ]:

# ============================================================
# STEP 3 — VALIDATE BEHAVIORAL WINDOWS FOR Fmr1
#
# Input:
#   Fmr1_all_behavior_windows.csv
#   Fmr1_all_trial_audit.csv
#
# Output:
#   Fmr1_behavior_windows_QC.csv
#   Fmr1_behavior_windows_analysis_v1.csv
#   Fmr1_behavior_phase_QC_summary.csv
#
# No electrophysiological metrics are calculated yet.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

RESULTS = ROOT / "04_results" / "Fmr1"
FIGURES = ROOT / "05_figures" / "Fmr1" / "Pilot_timelines" / "PNG"

FIGURES.mkdir(parents=True, exist_ok=True)

windows = pd.read_csv(
    RESULTS / "Fmr1_all_behavior_windows.csv"
)

audit = pd.read_csv(
    RESULTS / "Fmr1_all_trial_audit.csv"
)

# ------------------------------------------------------------
# 1. IDENTIFY TRIALS WITH ACTUAL TASK-RELATED MOVEMENT
# ------------------------------------------------------------

key = ["eid", "trial_id"]

movement_trials = (
    windows.loc[
        windows["phase"] == "M_WHEEL_MOVEMENT",
        key
    ]
    .drop_duplicates()
    .assign(has_motor_window=True)
)

windows_qc = windows.merge(
    movement_trials,
    on=key,
    how="left"
)

windows_qc["has_motor_window"] = (
    windows_qc["has_motor_window"]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# 2. CREATE EXPLICIT QC FLAGS
# ------------------------------------------------------------

windows_qc["qc_status"] = "accepted"
windows_qc["qc_reason"] = ""

# Nominal quiescence with detected wheel movement.
q_contaminated = (
    (windows_qc["phase"] == "Q_QUIESCENCE") &
    (windows_qc["movement_overlap_s"].fillna(0) > 0)
)

windows_qc.loc[q_contaminated, "qc_status"] = "flagged"
windows_qc.loc[q_contaminated, "qc_reason"] = (
    "detected_wheel_movement_during_nominal_quiescence"
)

# Stimulus windows without a qualifying motor-response window.
# They cannot be interpreted unambiguously as premovement.
s_no_motor = (
    (windows_qc["phase"] == "S_STIM_PREMOVEMENT") &
    (~windows_qc["has_motor_window"])
)

windows_qc.loc[s_no_motor, "qc_status"] = "flagged"
windows_qc.loc[s_no_motor, "qc_reason"] = (
    "no_qualifying_motor_response"
)

# Feedback is retained, with reward and error analyzed separately.
feedback = windows_qc["phase"] == "F_FEEDBACK"

invalid_feedback = (
    feedback &
    (~windows_qc["feedback_type"].isin([
        "correct_reward",
        "error_feedback"
    ]))
)

windows_qc.loc[invalid_feedback, "qc_status"] = "flagged"
windows_qc.loc[invalid_feedback, "qc_reason"] = (
    "unknown_feedback_outcome"
)

# Flag any interval extending beyond its nominal trial start
# only through the existing audit, not by arbitrary truncation.
# Here we preserve all original timestamps for review.

# ------------------------------------------------------------
# 3. SAVE COMPLETE QC TABLE AND ANALYSIS SUBSET
# ------------------------------------------------------------

windows_qc.to_csv(
    RESULTS / "Fmr1_behavior_windows_QC.csv",
    index=False
)

analysis_windows = windows_qc.loc[
    windows_qc["qc_status"] == "accepted"
].copy()

analysis_windows.to_csv(
    RESULTS / "Fmr1_behavior_windows_analysis_v1.csv",
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARY BY PHASE
# ------------------------------------------------------------

summary = (
    windows_qc.groupby("phase", sort=False)
    .agg(
        all_windows=("duration_s", "size"),
        accepted=("qc_status", lambda x: (x == "accepted").sum()),
        flagged=("qc_status", lambda x: (x == "flagged").sum()),
        total_duration_s=("duration_s", "sum"),
        median_duration_s=("duration_s", "median")
    )
    .reset_index()
)

summary["accepted_pct"] = (
    summary["accepted"] / summary["all_windows"] * 100
)

summary.to_csv(
    RESULTS / "Fmr1_behavior_phase_QC_summary.csv",
    index=False
)

print("=" * 70)
print("Fmr1 BEHAVIORAL WINDOW QC")
print("=" * 70)

print("Mice:", windows_qc["subject"].nunique())
print("Sessions:", windows_qc["eid"].nunique())
print("Original windows:", len(windows_qc))
print("Accepted:", len(analysis_windows))
print("Flagged:", (windows_qc["qc_status"] == "flagged").sum())

display(summary.round(3))

print("\nREASONS FOR FLAGGING")
display(
    windows_qc.loc[
        windows_qc["qc_status"] == "flagged",
        "qc_reason"
    ].value_counts().to_frame("windows")
)

print("\nFEEDBACK OUTCOMES — ACCEPTED")
display(
    analysis_windows.loc[
        analysis_windows["phase"] == "F_FEEDBACK",
        "feedback_type"
    ].value_counts().to_frame("windows")
)

# ------------------------------------------------------------
# 5. IMPROVED DURATION VISUALIZATION
# ------------------------------------------------------------

phase_names = {
    "Q_QUIESCENCE": "Quiescence",
    "S_STIM_PREMOVEMENT": "Stimulus / premovement",
    "M_WHEEL_MOVEMENT": "Wheel movement",
    "F_FEEDBACK": "Feedback",
    "I_INTERTRIAL": "Inter-trial interval"
}

phase_colors = {
    "Q_QUIESCENCE": "#8DA0CB",
    "S_STIM_PREMOVEMENT": "#66C2A5",
    "M_WHEEL_MOVEMENT": "#FC8D62",
    "F_FEEDBACK": "#B48BC8",
    "I_INTERTRIAL": "#9AABB9"
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 8),
    constrained_layout=True
)

plot_phases = [
    "Q_QUIESCENCE",
    "S_STIM_PREMOVEMENT",
    "M_WHEEL_MOVEMENT",
    "I_INTERTRIAL"
]

for ax, phase in zip(axes.flat, plot_phases):

    x = analysis_windows.loc[
        analysis_windows["phase"] == phase,
        "duration_s"
    ].dropna().to_numpy()

    x = x[np.isfinite(x)]

    if not len(x):
        ax.set_title(f"{phase_names[phase]} — no accepted windows")
        continue

    median = np.median(x)
    p99 = np.percentile(x, 99)
    upper = max(p99, np.min(x) + 0.001)

    # Only restrict the visualization, not the underlying data.
    x_display = x[x <= upper]

    ax.hist(
        x_display,
        bins=45,
        color=phase_colors[phase],
        edgecolor="white",
        linewidth=0.4
    )

    ax.axvline(
        median,
        color="#303030",
        linestyle="--",
        linewidth=1.6,
        label=f"Median = {median:.3f} s"
    )

    ax.set_title(
        phase_names[phase],
        fontsize=12,
        fontweight="bold"
    )

    ax.set_xlabel("Window duration (s)")
    ax.set_ylabel("Number of windows")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis="y", alpha=0.15)

fig.suptitle(
    "Fmr1 behavioral windows — accepted intervals",
    fontsize=15,
    fontweight="bold"
)

fig.savefig(
    FIGURES / "Fmr1_behavior_window_durations_QC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nQC TABLE:", RESULTS / "Fmr1_behavior_windows_QC.csv")
print("ANALYSIS WINDOWS:",
      RESULTS / "Fmr1_behavior_windows_analysis_v1.csv")


## Stage 3 — Extract Fmr1 spike metrics and save per-session caches (computationally expensive)


In [ ]:

# ============================================================
# STEP 5 — Fmr1 EVENT-ALIGNED ELECTROPHYSIOLOGY
#
# 15 Fmr1 mice | 9 regions | 10 electrophysiological metrics
#
# Outputs:
# - Per-window metrics (Parquet, by session)
# - Per-neuron and behavioral state metrics
# - Per-mouse / anatomical region metrics
# - Group summary and visualization
#
# Burst detection:
# Existing neuron-specific logISI threshold, maximum 100 ms,
# minimum 3 spikes per burst, no burst extension.
#
# Independent biological unit: mouse
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display
from brainbox.io.one import SpikeSortingLoader

ROOT = Path("/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned")

DATA = ROOT / "03_processed_data" / "Fmr1" / "Fmr1_all_neurons.csv"
BEHAV = ROOT / "04_results" / "Fmr1"

OUT = BEHAV / "electrophysiology_v1"
SESSION_OUT = OUT / "sessions"

FIG = ROOT / "05_figures" / "Fmr1" / "electrophysiology_v1"
FIG_PNG = FIG / "PNG"
FIG_PDF = FIG / "PDF"

for folder in [OUT, SESSION_OUT, FIG_PNG, FIG_PDF]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

MIN_STIM_S = 0.05
MAX_STIM_S = 5.0
MIN_MOVEMENT_S = 0.05

MIN_MICE = 5

# Save individual window metrics (one row per neuron/window).
SAVE_WINDOW_LEVEL = True

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

METRICS = [
    "fr_hz",
    "mean_isi_ms",
    "cv_isi",
    "cv2",
    "burst_rate_min",
    "burst_fraction_pct",
    "burst_duration_ms",
    "spikes_per_burst",
    "intraburst_hz",
    "interburst_interval_s"
]

STATES = [
    "Q",
    "S",
    "M",
    "F_reward",
    "F_error",
    "I"
]

STATE_LABELS = {
    "Q": "Quiescence",
    "S": "Stimulus",
    "M": "Movement",
    "F_reward": "Reward",
    "F_error": "Error",
    "I": "Inter-trial"
}

STATE_COLORS = {
    "Q": "#8298C6",
    "S": "#61AD94",
    "M": "#E39A5B",
    "F_reward": "#B38AC7",
    "F_error": "#D47C89",
    "I": "#899CAA"
}

# ------------------------------------------------------------
# 1. LOAD AND FILTER EXISTING BEHAVIORAL WINDOWS
# ------------------------------------------------------------

neurons = pd.read_csv(DATA)

windows = pd.read_csv(
    BEHAV / "Fmr1_behavior_windows_analysis_v1.csv"
)

# Previously flagged windows are absent from this table.
windows = windows.copy()

# Remove long or extremely short stimulus windows.
is_stim = windows["phase"].eq("S_STIM_PREMOVEMENT")

bad_stim = (
    is_stim &
    (
        (windows["duration_s"] < MIN_STIM_S) |
        (windows["duration_s"] > MAX_STIM_S)
    )
)

# Remove very short detected movement episodes.
bad_move = (
    windows["phase"].eq("M_WHEEL_MOVEMENT") &
    (windows["duration_s"] < MIN_MOVEMENT_S)
)

excluded = windows.loc[bad_stim | bad_move].copy()

windows = windows.loc[
    ~(bad_stim | bad_move)
].copy()

# Split feedback according to behavioral outcome.
state_map = {
    "Q_QUIESCENCE": "Q",
    "S_STIM_PREMOVEMENT": "S",
    "M_WHEEL_MOVEMENT": "M",
    "I_INTERTRIAL": "I"
}

windows["state"] = windows["phase"].map(state_map)

feedback_mask = windows["phase"].eq("F_FEEDBACK")

windows.loc[
    feedback_mask &
    windows["feedback_type"].eq("correct_reward"),
    "state"
] = "F_reward"

windows.loc[
    feedback_mask &
    windows["feedback_type"].eq("error_feedback"),
    "state"
] = "F_error"

windows = windows.loc[
    windows["state"].isin(STATES)
].copy()

# Preserve trial and window identity.
windows = windows.reset_index(drop=True)
windows["window_id"] = np.arange(len(windows))

windows = windows.sort_values(
    ["eid", "start_s", "end_s"]
).reset_index(drop=True)

windows.to_csv(
    OUT / "Fmr1_final_behavior_windows_v1.csv",
    index=False
)

excluded.to_csv(
    OUT / "Fmr1_additionally_excluded_windows_v1.csv",
    index=False
)

print("=" * 65)
print("INPUT DATA")
print("=" * 65)

print("Fmr1 neurons:", len(neurons))
print("Fmr1 mice:", neurons["subject"].nunique())
print("Sessions with windows:", windows["eid"].nunique())
print("Accepted windows:", len(windows))
print("Additionally excluded:", len(excluded))

display(windows["state"].value_counts().reindex(STATES))


# ------------------------------------------------------------
# 2. ELECTROPHYSIOLOGICAL METRICS
# ------------------------------------------------------------

def calculate_metrics(segments, threshold_ms):
    """
    Calculate metrics from multiple disjoint spike segments.

    Each segment is a tuple:
        (spike_times_in_seconds, window_duration_seconds)

    ISIs and bursts never cross segment boundaries.
    """

    n_spikes = 0
    total_duration = 0.0

    isi_parts = []
    cv2_parts = []

    burst_count = 0
    burst_spikes = 0

    burst_durations = []
    burst_sizes = []
    burst_frequencies = []
    interburst_intervals = []

    threshold_valid = (
        np.isfinite(threshold_ms) and
        threshold_ms > 0
    )

    threshold_s = (
        min(float(threshold_ms), 100.0) / 1000
        if threshold_valid else np.nan
    )

    for times, duration in segments:

        times = np.asarray(times, dtype=float)
        duration = float(duration)

        if not np.isfinite(duration) or duration <= 0:
            continue

        total_duration += duration
        n_spikes += len(times)

        if len(times) < 2:
            continue

        isi = np.diff(times)

        # Valid ISIs only within the same behavioral window.
        good_isi = isi[np.isfinite(isi) & (isi > 0)]

        if len(good_isi):
            isi_parts.append(good_isi)

        # Local CV2 requires consecutive positive ISIs.
        if len(isi) >= 2:

            a = isi[:-1]
            b = isi[1:]

            valid = (
                np.isfinite(a) &
                np.isfinite(b) &
                (a > 0) &
                (b > 0)
            )

            if np.any(valid):

                local_cv2 = (
                    2 * np.abs(a[valid] - b[valid]) /
                    (a[valid] + b[valid])
                )

                cv2_parts.append(local_cv2)

        if not threshold_valid:
            continue

        # Original modified logISI-core detector.
        short = (
            (isi > 0) &
            (isi <= threshold_s)
        )

        starts = np.flatnonzero(
            short & ~np.r_[False, short[:-1]]
        )

        ends = (
            np.flatnonzero(
                short & ~np.r_[short[1:], False]
            ) + 1
        )

        valid_bursts = (
            ends - starts + 1 >= 3
        )

        starts = starts[valid_bursts]
        ends = ends[valid_bursts]

        n_b = len(starts)

        if n_b == 0:
            continue

        counts = ends - starts + 1
        durations = times[ends] - times[starts]

        burst_count += n_b
        burst_spikes += int(counts.sum())

        burst_durations.extend(durations.tolist())
        burst_sizes.extend(counts.tolist())

        positive_duration = durations > 0

        if np.any(positive_duration):

            frequencies = (
                (counts[positive_duration] - 1) /
                durations[positive_duration]
            )

            burst_frequencies.extend(
                frequencies.tolist()
            )

        # Interburst intervals are only calculated
        # between bursts within the same window.
        if n_b > 1:

            intervals = (
                times[starts[1:]] -
                times[ends[:-1]]
            )

            interburst_intervals.extend(
                intervals.tolist()
            )

    result = {
        "n_spikes": n_spikes,
        "duration_s": total_duration,
        "n_bursts": burst_count,
        "n_valid_isi": sum(map(len, isi_parts)),
        "n_valid_cv2": sum(map(len, cv2_parts))
    }

    result.update({m: np.nan for m in METRICS})

    if total_duration <= 0:
        return result

    result["fr_hz"] = n_spikes / total_duration

    if isi_parts:

        all_isi = np.concatenate(isi_parts)

        result["mean_isi_ms"] = (
            all_isi.mean() * 1000
        )

        if len(all_isi) >= 2 and all_isi.mean() > 0:
            result["cv_isi"] = (
                all_isi.std(ddof=1) /
                all_isi.mean()
            )

    if cv2_parts:

        all_cv2 = np.concatenate(cv2_parts)
        result["cv2"] = all_cv2.mean()

    if threshold_valid:

        result["burst_rate_min"] = (
            burst_count * 60 / total_duration
        )

        if n_spikes > 0:
            result["burst_fraction_pct"] = (
                100 * burst_spikes / n_spikes
            )

        if burst_count > 0:

            result["burst_duration_ms"] = (
                np.mean(burst_durations) * 1000
            )

            result["spikes_per_burst"] = (
                np.mean(burst_sizes)
            )

            if burst_frequencies:
                result["intraburst_hz"] = (
                    np.mean(burst_frequencies)
                )

            if interburst_intervals:
                result["interburst_interval_s"] = (
                    np.mean(interburst_intervals)
                )

    return result


# ------------------------------------------------------------
# 3. PROCESS ALL Fmr1 SESSIONS
# ------------------------------------------------------------

session_neuron_results = []

session_ids = windows["eid"].astype(str).unique()

for session_no, eid in enumerate(session_ids, 1):

    session_windows = windows.loc[
        windows["eid"].astype(str) == eid
    ].copy()

    session_neurons = neurons.loc[
        neurons["eid"].astype(str) == eid
    ].copy()

    if session_neurons.empty:
        continue

    result_path = (
        SESSION_OUT / f"{eid}_neuron_phase.csv"
    )

    window_path = (
        SESSION_OUT / f"{eid}_window_metrics.parquet"
    )

    # Resume from successfully saved session results.
    if result_path.exists() and (
        not SAVE_WINDOW_LEVEL or window_path.exists()
    ):

        session_neuron_results.append(
            pd.read_csv(result_path)
        )

        print(
            f"[{session_no}/{len(session_ids)}] "
            f"Cached: {eid[:8]}"
        )

        continue

    current_neurons = []
    current_windows = []

    try:

        # Load each probe only once per session.
        for pid, units in session_neurons.groupby("pid"):

            loader = SpikeSortingLoader(
                pid=str(pid),
                one=one
            )

            spikes, clusters, channels = (
                loader.load_spike_sorting()
            )

            spike_times = np.asarray(
                spikes["times"], dtype=float
            )

            spike_clusters = np.asarray(
                spikes["clusters"]
            )

            # Group spikes by cluster ID.
            order = np.argsort(
                spike_clusters,
                kind="stable"
            )

            sorted_clusters = spike_clusters[order]
            sorted_times = spike_times[order]

            ids, starts, counts = np.unique(
                sorted_clusters,
                return_index=True,
                return_counts=True
            )

            cluster_ranges = {
                int(cid): (
                    int(start),
                    int(count)
                )
                for cid, start, count
                in zip(ids, starts, counts)
            }

            # Sort behavioral windows by start time.
            sw = session_windows.sort_values(
                "start_s"
            )

            w_start = sw["start_s"].to_numpy(float)
            w_end = sw["end_s"].to_numpy(float)

            for _, neuron in units.iterrows():

                cid = int(neuron["cluster_id"])

                if cid not in cluster_ranges:
                    continue

                start, count = cluster_ranges[cid]

                times = np.sort(
                    sorted_times[start:start+count]
                )

                if not len(times):
                    continue

                # Use the existing neuron-specific threshold.
                threshold = neuron.get(
                    "logisi_threshold_ms", np.nan
                )

                # Only windows within the spike recording
                # are used for this neuron.
                valid = (
                    (w_start >= times[0]) &
                    (w_end <= times[-1])
                )

                nw = sw.loc[valid].copy()

                if nw.empty:
                    continue

                a_idx = np.searchsorted(
                    times,
                    nw["start_s"].to_numpy(float),
                    side="left"
                )

                b_idx = np.searchsorted(
                    times,
                    nw["end_s"].to_numpy(float),
                    side="left"
                )

                # Collect segments by behavioral state.
                by_state = {
                    state: [] for state in STATES
                }

                for j, (_, window) in enumerate(nw.iterrows()):

                    segment = times[
                        a_idx[j]:b_idx[j]
                    ]

                    duration = float(
                        window["duration_s"]
                    )

                    state = window["state"]

                    by_state[state].append(
                        (segment, duration)
                    )

                    if SAVE_WINDOW_LEVEL:

                        measurements = calculate_metrics(
                            [(segment, duration)],
                            threshold
                        )

                        current_windows.append({
                            "subject": neuron["subject"],
                            "eid": eid,
                            "pid": pid,
                            "cluster_id": cid,
                            "region": neuron["region"],
                            "trial_id": window["trial_id"],
                            "window_id": window["window_id"],
                            "state": state,
                            "start_s": window["start_s"],
                            "end_s": window["end_s"],
                            **measurements
                        })

                # Pool all disjoint windows of each state.
                for state, segments in by_state.items():

                    if not segments:
                        continue

                    measurements = calculate_metrics(
                        segments,
                        threshold
                    )

                    current_neurons.append({
                        "subject": neuron["subject"],
                        "genotype": "Fmr1",
                        "eid": eid,
                        "pid": pid,
                        "cluster_id": cid,
                        "region": neuron["region"],
                        "state": state,
                        "n_windows": len(segments),
                        "logisi_threshold_ms": threshold,
                        **measurements
                    })

        if not current_neurons:
            raise RuntimeError("No matching neurons or windows")

        session_df = pd.DataFrame(current_neurons)

        # Write window-level data first.
        if SAVE_WINDOW_LEVEL:

            window_df = pd.DataFrame(current_windows)

            window_df.to_parquet(
                window_path,
                index=False,
                compression="snappy"
            )

        # Write session completion marker last.
        session_df.to_csv(
            result_path,
            index=False
        )

        session_neuron_results.append(session_df)

        print(
            f"[{session_no}/{len(session_ids)}] "
            f"{eid[:8]} | "
            f"{len(session_neurons)} neurons | "
            f"{len(session_df)} neuron-state records"
        )

    except Exception as exc:

        print(
            f"[{session_no}/{len(session_ids)}] "
            f"ERROR {eid[:8]}: {exc}"
        )

# ------------------------------------------------------------
# 4. COMBINE NEURON-LEVEL RESULTS
# ------------------------------------------------------------

if not session_neuron_results:
    raise RuntimeError("No neuron-level results were produced.")

neuron_phase = pd.concat(
    session_neuron_results,
    ignore_index=True
)

neuron_phase.to_csv(
    OUT / "Fmr1_neuron_phase_metrics.csv",
    index=False
)

print("\n" + "=" * 65)
print("NEURON-LEVEL RESULTS")
print("=" * 65)

print("Mice:", neuron_phase["subject"].nunique())
print("Sessions:", neuron_phase["eid"].nunique())

unique_units = neuron_phase[
    ["pid", "cluster_id"]
].drop_duplicates()

print("Unique neurons:", len(unique_units))
print("Neuron-state observations:", len(neuron_phase))

display(
    neuron_phase.groupby("state").agg(
        neurons=("cluster_id", "size"),
        total_duration_s=("duration_s", "sum")
    ).reindex(STATES).round(2)
)

# ------------------------------------------------------------
# 5. AGGREGATE NEURONS TO MOUSE x REGION x STATE
# ------------------------------------------------------------

# First aggregate repeated observations of the same neuron
# across sessions, if any, then average neurons within mice.
#
# Here each neuron is identified by insertion + cluster_id.
# Equal weighting across neurons is retained, as in the
# earlier spontaneous electrophysiology comparison.

mouse_region = (
    neuron_phase.groupby(
        ["subject", "region", "state"],
        as_index=False
    )
    .agg(
        n_neuron_records=("cluster_id", "size"),
        **{
            m: (m, "mean")
            for m in METRICS
        }
    )
)

mouse_region.to_csv(
    OUT / "Fmr1_mouse_region_phase_metrics.csv",
    index=False
)

# ------------------------------------------------------------
# 6. GROUP SUMMARY — ONLY STATES WITH >= 5 MICE
# ------------------------------------------------------------

summary_rows = []

for (region, state), subset in mouse_region.groupby(
    ["region", "state"]
):

    for metric in METRICS:

        x = subset[
            ["subject", metric]
        ].dropna()

        n = x["subject"].nunique()
        values = x[metric].to_numpy(float)

        summary_rows.append({
            "region": region,
            "state": state,
            "metric": metric,
            "n_mice": n,
            "mean": (
                values.mean() if n else np.nan
            ),
            "sd": (
                values.std(ddof=1)
                if n > 1 else np.nan
            ),
            "sem": (
                values.std(ddof=1) / np.sqrt(n)
                if n > 1 else np.nan
            ),
            "eligible_n5": n >= MIN_MICE
        })

group_summary = pd.DataFrame(summary_rows)

group_summary.to_csv(
    OUT / "Fmr1_group_region_phase_summary.csv",
    index=False
)

eligible = group_summary.loc[
    group_summary["eligible_n5"]
].copy()

eligible.to_csv(
    OUT / "Fmr1_group_region_phase_summary_n5.csv",
    index=False
)

print("\n" + "=" * 65)
print("GROUP-LEVEL SUMMARY")
print("=" * 65)

print("Region-state-metric combinations:",
      len(group_summary))
print("Eligible combinations (n >= 5):",
      len(eligible))

display(
    group_summary.pivot_table(
        index="region",
        columns="state",
        values="n_mice",
        aggfunc="max"
    ).reindex(index=REGIONS, columns=STATES)
)

# ------------------------------------------------------------
# 7. PLOT MOUSE-LEVEL FIRING RATE BY BEHAVIORAL STATE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3, 3,
    figsize=(17, 13),
    constrained_layout=True
)

rng = np.random.default_rng(42)

for ax, region in zip(axes.flat, REGIONS):

    subset = mouse_region.loc[
        mouse_region["region"] == region
    ]

    for j, state in enumerate(STATES):

        values = subset.loc[
            subset["state"] == state,
            "fr_hz"
        ].dropna().to_numpy(float)

        if len(values) < MIN_MICE:
            continue

        jitter = rng.uniform(
            -0.12, 0.12, len(values)
        )

        ax.scatter(
            np.full(len(values), j) + jitter,
            values,
            s=28,
            color=STATE_COLORS[state],
            alpha=0.75,
            edgecolor="white",
            linewidth=0.5
        )

        mean = values.mean()
        sem = (
            values.std(ddof=1) /
            np.sqrt(len(values))
        )

        ax.errorbar(
            j, mean, yerr=sem,
            fmt="o",
            color="black",
            markerfacecolor="white",
            capsize=4,
            markersize=7,
            linewidth=1.4
        )

    ax.set_title(
        region,
        fontweight="bold",
        fontsize=13
    )

    ax.set_xticks(range(len(STATES)))
    ax.set_xticklabels(
        STATES,
        fontsize=9
    )

    ax.set_ylabel("Firing rate (Hz)")
    ax.grid(axis="y", alpha=0.15)

fig.suptitle(
    "Fmr1 — Firing rate across behavioral states",
    fontsize=17,
    fontweight="bold"
)

fig.savefig(
    FIG_PNG / "fr_hz_Fmr1_all_regions.png",
    dpi=350,
    bbox_inches="tight"
)

fig.savefig(
    FIG_PDF / "fr_hz_Fmr1_all_regions.pdf",
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\n" + "=" * 65)
print("Fmr1 EVENT-ALIGNED ANALYSIS COMPLETE")
print("=" * 65)

print("Neuron-level:", OUT / "Fmr1_neuron_phase_metrics.csv")
print("Mouse-level:", OUT / "Fmr1_mouse_region_phase_metrics.csv")
print("Summary:", OUT / "Fmr1_group_region_phase_summary.csv")
print("Figures:", FIG)
